# CS1 EXP-1 -- Word/Char TF-IDF + SGD Logistic Regression (rotating 5-fold, nested hyperparameter search)

## 1. Runtime settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DRIVE_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

INPUT_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
ABSTRACTED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2.parquet"
DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
#CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP1_OUTPUT_DIR = OUTPUT_ROOT / f"exp1_lr_rotating5fold_{CODE_COLUMN_TAG}"

RUN_NORMALIZATION_TESTS = True
RUN_PROFILE_FOLD = False
RUN_OFFICIAL = True

print("Settings loaded.")
print("Downsampled parquet:", DOWNSAMPLED_PARQUET)
print("Manifest:", MANIFEST_PATH)
print("Output dir:", EXP1_OUTPUT_DIR)


Settings loaded.
Downsampled parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet
Manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_shared_rotating_5fold_v1/project_grouped_5fold_manifest.parquet
Output dir: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized


## 2. Fetch repository

In [2]:
import urllib.request
import zipfile


def download_and_extract_repo(repo_url, branch, target_dir):
    if target_dir.exists():
        print(f"Repository already exists at {target_dir}")
        return

    print(f"Downloading {repo_url} (branch: {branch}) without git...")
    clean_url = repo_url.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{branch}.zip"
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target_dir.parent / f"{target_dir.name}_download_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir.parent)

    repo_name = clean_url.split("/")[-1]
    extracted_folder = target_dir.parent / f"{repo_name}-{branch}"
    if extracted_folder.exists():
        extracted_folder.rename(target_dir)

    zip_path.unlink()
    print(f"Repository ready at {target_dir}")
import sys

download_and_extract_repo(REPO_URL, REPO_BRANCH, REPO_ROOT)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Missing project directory: {PROJECT_DIR}")
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Missing source directory: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nRepository ready.")
print("Repo root:", REPO_ROOT)
print("Project dir:", PROJECT_DIR)
print("Source dir:", SRC_DIR)


Repository already exists at /workspace/DiverseVul--IS-Project

Repository ready.
Repo root: /workspace/DiverseVul--IS-Project
Project dir: /workspace/DiverseVul--IS-Project/vuln-detection
Source dir: /workspace/DiverseVul--IS-Project/vuln-detection/src


## 3. Install/import dependencies and verify committed files

In [3]:

import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pyarrow", "joblib", "pytest"
], check=True)
print("Dependencies installed/verified.")


Dependencies installed/verified.


In [4]:
import json
import time
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd

required_repo_files = [
    SRC_DIR / "utils" / "normalization_v3.py",
    SRC_DIR / "utils" / "scope2_preprocessing.py",
    SRC_DIR / "utils" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "exp1" / "exp1_lr.py",
]

missing = [str(path) for path in required_repo_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required committed files:\n" + "\n".join(missing))

from utils.normalization_v3 import NORMALIZATION_VERSION
from utils import scope2_preprocessing as s2p
from utils import split_manifest
from case_study_1.exp1.exp1_lr import Exp1Config, run_exp1, run_exp1_profile_fold

print("Imported project modules successfully.")
print("NORMALIZATION_VERSION:", NORMALIZATION_VERSION)
print("ABSTRACTION_VERSION:", s2p.SCOPE2_ABSTRACTION_VERSION)


Imported project modules successfully.
NORMALIZATION_VERSION: cs1-conservative-v3-no-nfkc
ABSTRACTION_VERSION: cs1-scope2-abstraction-v2


In [5]:


from pathlib import Path

test_file = PROJECT_DIR / "tests" / "test_normalization_v3.py"

text = test_file.read_text(encoding="utf-8")

text = text.replace(
    "from normalization_v3 import",
    "from utils.normalization_v3 import",
)

test_file.write_text(text, encoding="utf-8")

print("Patched:", test_file)
print(test_file.read_text(encoding="utf-8").splitlines()[:10])

Patched: /workspace/DiverseVul--IS-Project/vuln-detection/tests/test_normalization_v3.py
['import pytest', 'import pandas as pd', '', 'from utils.normalization_v3 import (', '    normalize_code,', '    _normalize_line_endings_and_unicode,', '    iter_cpp_lexical_tokens,', ')', '', '']


## 4. Optional: run normalization v3 tests

In [6]:


import os
import subprocess
import sys
from pathlib import Path

if RUN_NORMALIZATION_TESTS:
    test_file = PROJECT_DIR / "tests" / "test_normalization_v3.py"

    if not test_file.is_file():
        print("Test file not found, skipping:", test_file)
    else:
        env = os.environ.copy()
        env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

        command = [
            sys.executable,
            "-m",
            "pytest",
            str(test_file),
            "-q",
            "-s",
        ]

        print("$", " ".join(str(x) for x in command))
        print("PYTHONPATH:", env["PYTHONPATH"])

        result = subprocess.run(
            command,
            cwd=str(PROJECT_DIR),
            env=env,
            text=True,
            capture_output=True,
        )

        print("\n--- pytest stdout ---")
        print(result.stdout)

        print("\n--- pytest stderr ---")
        print(result.stderr)

        if result.returncode != 0:
            raise RuntimeError(
                f"normalization_v3 tests failed with return code {result.returncode}"
            )

        print("normalization_v3 tests passed.")
else:
    print("RUN_NORMALIZATION_TESTS=False, skipping tests.")

$ /usr/bin/python3 -m pytest /workspace/DiverseVul--IS-Project/vuln-detection/tests/test_normalization_v3.py -q -s
PYTHONPATH: /workspace/DiverseVul--IS-Project/vuln-detection/src:

--- pytest stdout ---
....
4 passed in 0.38s


--- pytest stderr ---

normalization_v3 tests passed.


## 5. Load abstracted_code_v1 (built by scope2_preprocessing.ipynb)

In [7]:
if not INPUT_PARQUET.is_file():
    raise FileNotFoundError(f"Missing input parquet: {INPUT_PARQUET}")

if not ABSTRACTED_PARQUET.is_file():
    raise FileNotFoundError(
        f"Missing abstracted parquet: {ABSTRACTED_PARQUET}\n"
        "Run notebooks/scope2_preprocessing.ipynb first to build it."
    )

print("Abstracted parquet found:", ABSTRACTED_PARQUET)


Abstracted parquet found: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v2.parquet


In [8]:
preview_df = pd.read_parquet(ABSTRACTED_PARQUET, columns=["normalized_code", "abstracted_code_v1"])
print("Rows:", len(preview_df))
for _, row in preview_df.head(3).iterrows():
    print("-" * 80)
    print("normalized_code   :", str(row["normalized_code"])[:200])
    print("abstracted_code_v1:", str(row["abstracted_code_v1"])[:200])
del preview_df


Rows: 261667
--------------------------------------------------------------------------------
normalized_code   : void dcn20_calculate_wm(
struct dc *dc, struct dc_state *context,
display_e2e_pipe_params_st *pipes,
int *out_pipe_cnt,
int *pipe_split_from,
int vlevel)
{
int pipe_cnt, i, pipe_idx;
for (i = 0, pipe_
abstracted_code_v1: void FUNC0@TOKEN@(
struct VAR0@TOKEN@ *VAR0@TOKEN@, struct dc_state *VAR1@TOKEN@,
display_e2e_pipe_params_st *VAR2@TOKEN@,
int *VAR3@TOKEN@,
int *VAR4@TOKEN@,
int VAR5@TOKEN@)
{
int VAR6@TOKEN@, VAR7@
--------------------------------------------------------------------------------
normalized_code   : int hw_atl_utils_soft_reset(struct aq_hw_s *self)
{
int ver = hw_atl_utils_get_fw_version(self);
u32 boot_exit_code = 0;
u32 val;
int k;
for (k = 0; k < 1000; ++k) {
u32 flb_status = aq_hw_read_reg(se
abstracted_code_v1: int FUNC0@TOKEN@(struct aq_hw_s *VAR0@TOKEN@)
{
int VAR1@TOKEN@ = VAR2@TOKEN@(VAR0@TOKEN@);
u32 VAR3@TOKEN@ = 0;
u32 VAR4@TOKEN@;
int VAR5@TOK

## 6. Load the downsampled dataset and shared 5-fold manifest

In [9]:
if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(
        f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}\n"
        "Run notebooks/scope2_preprocessing.ipynb first to build it."
    )
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Missing manifest: {MANIFEST_PATH}\n"
        "Run notebooks/scope2_preprocessing.ipynb's manifest-generation section first."
    )

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(
    MANIFEST_PATH,
    config=split_manifest.SplitConfig(n_splits=5, random_state=42),
)

required_full_columns = {"source_row_id", "code", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_full = required_full_columns.difference(full_df.columns)
if missing_full:
    raise KeyError(f"Downsampled dataset missing columns: {sorted(missing_full)}")

print("Downsampled dataset:", full_df.shape)
print("Manifest:", manifest_df.shape)

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


Downsampled dataset: (20000, 7)
Manifest: (20000, 4)


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,3473,198,3275,0.057011,-0.002039,151,16527,615,0,0.17365,0.197128
1,1,4383,268,4115,0.061145,0.002095,156,15617,610,0,0.21915,0.203655
2,2,4341,224,4117,0.051601,-0.007449,156,15659,610,0,0.21705,0.203655
3,3,4092,253,3839,0.061828,0.002778,151,15908,615,0,0.20460,0.197128
4,4,3711,238,3473,0.064134,0.005084,152,16289,614,0,0.18555,0.198433


Fold test-size balance: smallest=3473 rows, largest=4383 rows, ratio=1.26x
Fold sizes look reasonably balanced.


## 7. Manifest validation and empty-code guard

In [10]:
if full_df["source_row_id"].duplicated().any():
    raise RuntimeError("full_df contains duplicate source_row_id values.")
if manifest_df["source_row_id"].duplicated().any():
    raise RuntimeError("manifest_df contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if not manifest_ids.issubset(set(full_indexed.index)):
    raise RuntimeError("Manifest contains IDs not present in the downsampled dataset.")
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

joined = manifest_df.set_index("source_row_id").join(full_indexed[["label", "project"]], rsuffix="_dataset")
if not (joined["label"].astype(int) == joined["label_dataset"].astype(int)).all():
    raise RuntimeError("Label mismatch between downsampled dataset and manifest.")
if not (joined["project"].astype(str) == joined["project_dataset"].astype(str)).all():
    raise RuntimeError("Project mismatch between downsampled dataset and manifest.")

dataset_frame = full_indexed.loc[list(manifest_ids)].copy().reset_index(drop=True)
dataset_frame = dataset_frame[
    ["source_row_id", "code", "abstracted_code_v1", "normalized_code", "label", "project"]
].copy()

EMPTY_CODE_SENTINEL = "EMPTY_ABSTRACTED_CODE_SAMPLE"
dataset_frame[CODE_COLUMN] = dataset_frame[CODE_COLUMN].fillna("").astype(str)
empty_mask = dataset_frame[CODE_COLUMN].str.strip().eq("")
n_empty = int(empty_mask.sum())
if n_empty:
    dataset_frame.loc[empty_mask, CODE_COLUMN] = EMPTY_CODE_SENTINEL
    print(f"Replaced {n_empty} empty {CODE_COLUMN} rows with sentinel token.")
assert not dataset_frame[CODE_COLUMN].str.strip().eq("").any()

print("Dataset ready:", dataset_frame.shape, "| positive rate:", dataset_frame["label"].mean())
print("Unique projects:", dataset_frame["project"].nunique())


Dataset ready: (20000, 6) | positive rate: 0.05905
Unique projects: 766


## 8. Configure EXP-1

In [11]:
config = Exp1Config(
    experiment_name=f"cs1_exp1_lr_{CODE_COLUMN_TAG}",
    code_column=CODE_COLUMN,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    n_splits=5,
    random_state=42,
    top_features_per_direction=15,
    verbose=True,
)

print("Code column:", config.code_column)
print("word_max_features_grid:", config.word_max_features_grid)
print("char_max_features_grid:", config.char_max_features_grid)
print("word_min_df_grid:", config.word_min_df_grid)
print("char_min_df_grid:", config.char_min_df_grid)
print("sgd_alpha_grid:", config.sgd_alpha_grid)
print("Output directory:", EXP1_OUTPUT_DIR)


Code column: normalized_code
word_max_features_grid: (20000, 50000, 80000)
char_max_features_grid: (30000, 60000, 90000)
word_min_df_grid: (2, 3)
char_min_df_grid: (5, 8)
sgd_alpha_grid: (1e-06, 3e-06, 1e-05, 3e-05, 0.0001)
Output directory: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized


## 9. Optional profile run -- inner search + refit for one outer fold

In [12]:
if RUN_PROFILE_FOLD:
    profile = run_exp1_profile_fold(
        normalized_frame=dataset_frame,
        manifest=manifest_df,
        fold_id=4,
        config=config,
    )
    print("Selected hyperparameters:")
    display(pd.DataFrame([profile["selection"]]))
    print("Fold metrics:")
    display(profile["profile_metrics"])
else:
    print("RUN_PROFILE_FOLD=False; skipping profile.")


RUN_PROFILE_FOLD=False; skipping profile.


## 10. Official rotating 5-fold run

In [13]:
def _resolve_repo_commit(repo_root: Path, repo_branch: str) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL,
        ).decode().strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return f"unknown (repo fetched via zip archive, branch={repo_branch}, no .git metadata)"


if RUN_OFFICIAL:
    results = run_exp1(
        normalized_frame=dataset_frame,
        manifest=manifest_df,
        config=config,
        output_dir=EXP1_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "input_column": CODE_COLUMN,
            "abstraction_version": s2p.SCOPE2_ABSTRACTION_VERSION,
            "normalization_version": NORMALIZATION_VERSION,
            "manifest_path": str(MANIFEST_PATH),
            "repo_commit": _resolve_repo_commit(REPO_ROOT, REPO_BRANCH),
        },
    )
    print("\nOfficial EXP-1 run complete.")
else:
    results = None
    print("RUN_OFFICIAL=False; official training skipped.")


[13:16:07] CS1-EXP1 official run started: 5 rotating outer folds.
[13:16:07] Outer fold 1/5 | inner search started (3-fold project-grouped CV; 36 TF-IDF combos x 5 alpha values).
[13:29:03] Outer fold 1/5, inner fold 1/3 | done.
[13:41:42] Outer fold 1/5, inner fold 2/3 | done.
[13:54:30] Outer fold 1/5, inner fold 3/3 | done.
[13:54:32] Outer fold 1/5 | selected word_max_features=80000, char_max_features=60000, word_min_df=2, char_min_df=8, alpha=0.0001, threshold=0.46 (inner PR-AUC=0.1421).
[13:54:55] Outer fold 1/5 | fit+scored in 22.9s.
[13:54:55] Outer fold 2/5 | inner search started (3-fold project-grouped CV; 36 TF-IDF combos x 5 alpha values).
[14:06:59] Outer fold 2/5, inner fold 1/3 | done.
[14:18:56] Outer fold 2/5, inner fold 2/3 | done.
[14:30:51] Outer fold 2/5, inner fold 3/3 | done.
[14:30:52] Outer fold 2/5 | selected word_max_features=80000, char_max_features=90000, word_min_df=3, char_min_df=8, alpha=0.0001, threshold=0.47 (inner PR-AUC=0.1486).
[14:31:15] Outer fold

## 11. Display result summary

In [14]:
if results is None:
    print("No official results object in memory. Set RUN_OFFICIAL=True.")
else:
    pooled = results["evaluation"]["pooled_metrics"]
    fold_metrics = results["evaluation"]["fold_metrics"]
    fold_summary = results["evaluation"]["fold_summary"]
    fold_training = results["fold_training"]

    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in pooled.items()]))

    print("Per-fold metrics:")
    display(fold_metrics)

    print("Mean +/- std across the 5 outer folds (headline result):")
    display(fold_summary)

    print("Selected hyperparameters by outer fold:")
    display(fold_training[[
        "fold", "word_max_features", "char_max_features", "word_min_df", "char_min_df",
        "sgd_alpha", "decision_threshold",
    ]])

    print("Artifacts:", results.get("artifacts"))


Pooled OOF metrics (secondary cross-check):


,metric,value
0,n_samples,20000.000000
1,vulnerable_1,1181.000000
2,non_vulnerable_0,18819.000000
3,positive_rate,0.059050
4,threshold,0.498000
5,average_precision_pr_auc,0.145306
6,precision,0.155597
7,recall,0.353091
8,f1,0.216006
9,mcc,0.161116


Per-fold metrics:


,fold,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,...,negative_predictive_value,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,test_unique_projects
0,0,3473,198,3275,0.057011,0.498,0.142710,0.165432,0.338384,0.222222,...,0.957301,0.103206,0.661616,2937,338,131,67,405,0.116614,151
1,1,4383,268,4115,0.061145,0.498,0.148997,0.142355,0.302239,0.193548,...,0.950970,0.118591,0.697761,3627,488,187,81,569,0.129820,156
2,2,4341,224,4117,0.051601,0.498,0.148479,0.149856,0.464286,0.226580,...,0.967096,0.143308,0.535714,3527,590,120,104,694,0.159871,156
3,3,4092,253,3839,0.061828,0.498,0.169636,0.170498,0.351779,0.229677,...,0.954062,0.112790,0.648221,3406,433,164,89,522,0.127566,151
4,4,3711,238,3473,0.064134,0.498,0.137201,0.155102,0.319328,0.208791,...,0.949705,0.119205,0.680672,3059,414,162,76,490,0.132040,152


Mean +/- std across the 5 outer folds (headline result):


,metric,mean,std,min,max
0,positive_rate,0.059144,0.004938,0.051601,0.064134
1,threshold,0.498000,0.000000,0.498000,0.498000
2,average_precision_pr_auc,0.149405,0.012289,0.137201,0.169636
3,precision,0.156649,0.011421,0.142355,0.170498
4,recall,0.355203,0.063808,0.302239,0.464286
5,f1,0.216164,0.014947,0.193548,0.229677
6,mcc,0.162384,0.024707,0.130916,0.193750
7,accuracy,0.849258,0.010780,0.836443,0.864958
8,balanced_accuracy,0.617891,0.026526,0.591824,0.660489
9,specificity,0.880580,0.014817,0.856692,0.896794


Selected hyperparameters by outer fold:


,fold,word_max_features,char_max_features,word_min_df,char_min_df,sgd_alpha,decision_threshold
0,0,80000,60000,2,8,0.0001,0.46
1,1,80000,90000,3,8,0.0001,0.47
2,2,80000,90000,3,5,0.0001,0.45
3,3,80000,30000,3,5,0.0001,0.59
4,4,80000,90000,3,5,0.0001,0.52


Artifacts: Exp1Artifacts(config_json=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized/cs1_exp1_lr_normalized_config.json'), fold_training_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized/cs1_exp1_lr_normalized_fold_training.csv'), inner_search_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized/cs1_exp1_lr_normalized_inner_search.csv'), top_features_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized/cs1_exp1_lr_normalized_top_features.csv'), run_metadata_json=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp1_lr_rotating5fold_normalized/cs1_exp1_lr_normalized_run_metadata.json'), evaluation_paths=EvaluationPaths(predictions_parquet=PosixPath('/workspace/IntelligentSystemProject/Vulner

## 11b. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
import utils.confidence_intervals as confidence_intervals

exp1_oof_ci = confidence_intervals.bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp1_oof_ci))


## 12. Interpretability: top TF-IDF weights per fold

In [ ]:
top_features = results["top_features"]
for fold_id in sorted(top_features["fold"].unique()):
    print(f"--- Fold {fold_id} ---")
    fold_top = top_features[top_features["fold"] == fold_id]
    print("Top vulnerable-associated:")
    display(fold_top[fold_top["direction"] == "vulnerable_associated"].head(10))
    print("Top non-vulnerable-associated:")
    display(fold_top[fold_top["direction"] == "non_vulnerable_associated"].head(10))

top_features.to_csv(EXP1_OUTPUT_DIR / "exp1_top_features_all_folds.csv", index=False)


## 13. Error analysis: pooled OOF false positives/negatives

In [ ]:
oof = results["oof_predictions"].merge(
    dataset_frame[["source_row_id", "code"]], on="source_row_id", how="left",
)

false_positives = oof[(oof["label"] == 0) & (oof["y_pred"] == 1)]
false_negatives = oof[(oof["label"] == 1) & (oof["y_pred"] == 0)]

print(f"Extracted {len(false_positives)} False Positives and {len(false_negatives)} False Negatives (pooled OOF, full dataset).")

sample_columns = ["source_row_id", "project", "y_score", "code"]
false_positives[sample_columns].sample(n=min(5, len(false_positives)), random_state=42).to_csv(
    EXP1_OUTPUT_DIR / "sample_false_positives.csv", index=False
)
false_negatives[sample_columns].sample(n=min(5, len(false_negatives)), random_state=42).to_csv(
    EXP1_OUTPUT_DIR / "sample_false_negatives.csv", index=False
)
